# Xây dựng Custom AlexNet (BatchNorm) cho Oxford-IIIT Pet từ đầu

**Yêu cầu xây dựng model:**
- Input: 224x224x3
- Classes: 37 giống loài (12 giống mèo, 25 giống chó) thuộc 2 loài động vật Chó và Mèo
- Framework: PyTorch
- Pretrained: Không sử dụng (From Scratch)

In [ ]:
# [Môi trường Kaggle / Colab] Tự động tải mã nguồn từ GitHub và chuyển vào đúng thư mục làm việc
import os

if os.path.exists("/kaggle"):
    %cd /kaggle/working
    if not os.path.exists("ml-semester-cat-dog-classifier"):
        !git clone https://github.com/LuanNguyen23T/ml-semester-cat-dog-classifier.git
    %cd /kaggle/working/ml-semester-cat-dog-classifier/notebooks/02_alexnet
elif os.path.exists("/content"):
    %cd /content
    if not os.path.exists("ml-semester-cat-dog-classifier"):
        !git clone https://github.com/LuanNguyen23T/ml-semester-cat-dog-classifier.git
    %cd /content/ml-semester-cat-dog-classifier/notebooks/02_alexnet


## Bước 1: Input các thư viện

In [ ]:
import os
import sys
import json
import random
import argparse
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from sklearn.metrics import confusion_matrix

# Thiết lập Random Seed chuẩn xác và Deterministic cho Card T4 GPU
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Tự động tìm thư mục gốc chứa package src (tương thích cả clone mới, Colab, Jupyter)
from pathlib import Path
current_path = Path.cwd()
project_root = None
for parent in [current_path, current_path.parent, current_path.parent.parent]:
    if (parent / "src").is_dir():
        project_root = parent
        break

if project_root is not None:
    for p in [str(project_root), str(project_root / "src")]:
        if p not in sys.path:
            sys.path.insert(0, p)

# Nạp modules: import chuẩn từ package src (IDE linter nhận diện ngay không cần file cấu hình)
try:
    from src.dataset import OxfordPetDataset, get_transforms, CLAHETransform
except ImportError:
    from dataset import OxfordPetDataset, get_transforms, CLAHETransform

device = torch.device('cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu'))
if torch.cuda.is_available():
    print(f"Thiết bị đang sử dụng: {torch.cuda.get_device_name(0)} (CUDA T4) | Seed: {SEED}")
else:
    print(f"Thiết bị đang sử dụng: {device} | Seed: {SEED}")

## Bước 2: Argparse & Thiết lập siêu tham số

In [ ]:
def parse_args():
    parser = argparse.ArgumentParser(description="Custom AlexNet (BatchNorm) (High Accuracy on T4)")
    # Cấu hình tối ưu độ chính xác trên GPU 16GB VRAM (Tesla T4)
    # Tự động định vị thư mục dataset trên mọi môi trường (Local / Colab)
    default_data_dir = "../../dataset/oxford-iiit-pet"
    if project_root and (project_root / "dataset" / "oxford-iiit-pet").exists():
        default_data_dir = str(project_root / "dataset" / "oxford-iiit-pet")
    elif project_root and (project_root / "dataset").exists():
        default_data_dir = str(project_root / "dataset" / "oxford-iiit-pet")
    parser.add_argument("--data_dir", type=str, default=default_data_dir)
    parser.add_argument("--task", type=str, default="breed", choices=["breed", "binary"])
    parser.add_argument("--img_size", type=int, default=224, help="Kích thước ảnh chuẩn 224x224")
    parser.add_argument("--batch_size", type=int, default=64, help="Batch size 64 tối ưu ước lượng gradient đa lớp trên T4")
    parser.add_argument("--epochs", type=int, default=50, help="Số epochs huấn luyện 50")
    parser.add_argument("--lr", type=float, default=5e-4, help="Learning rate tinh chỉnh 0.0005 cho AdamW")
    parser.add_argument("--weight_decay", type=float, default=1e-2, help="Decoupled weight decay 0.01 chống overfitting")
    parser.add_argument("--label_smoothing", type=float, default=0.1, help="Label smoothing 0.1 tăng khả năng tổng quát hóa")
    parser.add_argument("--checkpoint_dir", type=str, default="./checkpoints/baseline")
    parser.add_argument("--plots_dir", type=str, default="./plots/baseline")
    try:
        args, _ = parser.parse_known_args()
    except:
        args = parser.parse_args([])
    return args

args = parse_args()

## Bước 3: Load Data & Data Augmentation

In [ ]:
# Data Augmentation nâng cao: Extreme Part Crop (8% - 40% diện tích) + CLAHE LAB
train_transform = get_transforms(img_size=args.img_size, is_train=True, use_part_aug=True, use_clahe=True)
test_transform = get_transforms(img_size=args.img_size, is_train=False)

trainset = OxfordPetDataset(root_dir=args.data_dir, split="trainval", task=args.task, transform=train_transform)
trainloader = DataLoader(trainset, batch_size=args.batch_size, shuffle=True, num_workers=2, pin_memory=torch.cuda.is_available())

testset = OxfordPetDataset(root_dir=args.data_dir, split="test", task=args.task, transform=test_transform)
testloader = DataLoader(testset, batch_size=args.batch_size, shuffle=False, num_workers=2, pin_memory=torch.cuda.is_available())

classes = trainset.class_names if args.task == "breed" else ["Cat", "Dog"]
num_classes = len(classes)
print(f"Tổng số mẫu Train: {len(trainset):,} | Test: {len(testset):,} | Số lớp: {num_classes}")
print(f"Batch Size: {args.batch_size} | Kích thước ảnh: {args.img_size}x{args.img_size} px")

# Trích xuất và trực quan hóa bộ phận chính xác kết hợp Trimap (tách nền) + XML Ground Truth
import numpy as np
import xml.etree.ElementTree as ET
from PIL import ImageDraw

def extract_parts_trimap_xml(img_path, annotations_dir, target_size=(224, 224)):
    img = Image.open(img_path).convert("RGB")
    w, h = img.size
    base_name = os.path.splitext(os.path.basename(img_path))[0]
    
    xml_path = os.path.join(annotations_dir, "xmls", f"{base_name}.xml")
    trimap_path = os.path.join(annotations_dir, "trimaps", f"{base_name}.png")
    
    # 1. Trích xuất Head BBox từ XML nếu có
    head_box = None
    if os.path.exists(xml_path):
        try:
            tree = ET.parse(xml_path)
            bndbox = tree.getroot().find("object").find("bndbox")
            xmin = max(0, int(bndbox.find("xmin").text))
            ymin = max(0, int(bndbox.find("ymin").text))
            xmax = min(w, int(bndbox.find("xmax").text))
            ymax = min(h, int(bndbox.find("ymax").text))
            head_box = (xmin, ymin, xmax, ymax)
        except Exception:
            pass

    # 2. Trích xuất Mask cơ thể chó/mèo từ Trimap (1 = pet body, 3 = contour) để loại bỏ 100% nền nhà
    body_box = None
    if os.path.exists(trimap_path):
        try:
            tri_arr = np.array(Image.open(trimap_path))
            pet_mask = (tri_arr == 1) | (tri_arr == 3)
            
            if head_box is not None:
                # Lọc các pixel thuộc thân thể nằm phía dưới tâm đầu (loại bỏ hoàn toàn nền nhà)
                head_mid_y = int((head_box[1] + head_box[3]) / 2)
                lower_body_mask = pet_mask.copy()
                lower_body_mask[:head_mid_y, :] = False
                
                indices = np.argwhere(lower_body_mask)
                if len(indices) > 50:
                    b_ymin, b_xmin = indices.min(axis=0)
                    b_ymax, b_xmax = indices.max(axis=0)
                    body_box = (int(b_xmin), int(b_ymin), int(b_xmax), int(b_ymax))
            
            if body_box is None:
                indices = np.argwhere(pet_mask)
                if len(indices) > 50:
                    ymin_f, xmin_f = indices.min(axis=0)
                    ymax_f, xmax_f = indices.max(axis=0)
                    body_box = (int(xmin_f), int(ymin_f + 0.4 * (ymax_f - ymin_f)), int(xmax_f), int(ymax_f))
                    if head_box is None:
                        head_box = (int(xmin_f), int(ymin_f), int(xmax_f), int(ymin_f + 0.6 * (ymax_f - ymin_f)))
        except Exception:
            pass

    # 3. Fallback hình học nếu thiếu cả XML lẫn Trimap
    if head_box is None:
        head_box = (0, 0, w, int(h * 0.55))
    if body_box is None:
        body_box = (0, int(h * 0.40), w, h)

    head_crop = img.crop(head_box).resize(target_size, Image.Resampling.BILINEAR)
    body_crop = img.crop(body_box).resize(target_size, Image.Resampling.BILINEAR)

    # Đánh dấu Bounding Box trực quan trên ảnh gốc
    annotated_img = img.copy()
    draw = ImageDraw.Draw(annotated_img)
    draw.rectangle(head_box, outline="#06d6a0", width=4)   # Khung xanh: Đầu / Tai
    draw.rectangle(body_box, outline="#ef476f", width=4)   # Khung đỏ: Chân / Thân (Trimap sạch nền)

    return annotated_img, head_crop, body_crop

# Minh họa trực quan trên ảnh mẫu số 10
sample_idx = 10
sample_path, _, _, _, sample_name = trainset.samples[sample_idx]
ann_img, crop_head, crop_paw = extract_parts_trimap_xml(sample_path, os.path.join(args.data_dir, "annotations"), target_size=(args.img_size, args.img_size))
clahe_img = CLAHETransform(clip_limit=3.0, p=1.0)(Image.open(sample_path).convert("RGB"))

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(ann_img); axes[0].set_title(f"Ảnh gốc: {sample_name} | Xanh: Đầu, Đỏ: Chân", fontsize=10); axes[0].axis("off")
axes[1].imshow(crop_head); axes[1].set_title("Bộ phận: Tai / Đầu (Head)", fontsize=11, fontweight="bold"); axes[1].axis("off")
axes[2].imshow(crop_paw); axes[2].set_title("Bộ phận: Chân / Thân (Paws/Body)", fontsize=11, fontweight="bold"); axes[2].axis("off")
axes[3].imshow(clahe_img); axes[3].set_title("Lọc CLAHE Tăng Nét Lông", fontsize=11); axes[3].axis("off")
plt.tight_layout()
plt.show()


## Bước 4: Kiến trúc Model From Scratch

In [ ]:
class AlexNetBN(nn.Module):
    def __init__(self, num_classes=37, in_channels=3):
        super(AlexNetBN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 64, kernel_size=11, stride=4, padding=2, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),

            nn.Conv2d(64, 192, kernel_size=5, padding=2, bias=False),
            nn.BatchNorm2d(192),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),

            nn.Conv2d(192, 384, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(384),
            nn.ReLU(inplace=True),

            nn.Conv2d(384, 256, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),

            nn.Conv2d(256, 256, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((6, 6)),
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(256 * 6 * 6, 2048),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(2048, 1024),
            nn.ReLU(inplace=True),
            nn.Linear(1024, num_classes),
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)

    def forward(self, x):
        return self.classifier(self.features(x))

model = AlexNetBN(num_classes=num_classes).to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Đã chọn mô hình: Custom AlexNet (BatchNorm) (From Scratch)")
print(f"Khởi tạo trọng số Kaiming He Normal hoàn tất!")
print(f"Tổng số tham số: {total_params:,}")

## Bước 5: Huấn luyện (Training & Checkpoints)

In [ ]:
os.makedirs(args.checkpoint_dir, exist_ok=True)
# Sử dụng Label Smoothing CrossEntropyLoss để tối ưu hóa độ chính xác phân loại đa lớp
criterion = nn.CrossEntropyLoss(label_smoothing=args.label_smoothing)
optimizer = optim.AdamW(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=args.epochs, eta_min=1e-6)

best_ckpt = os.path.join(args.checkpoint_dir, "alexnet_best.pth")
if not os.path.exists(best_ckpt):
    for alt_p in [os.path.join(os.path.dirname(args.checkpoint_dir), "alexnet_best.pth"), os.path.join(".", "checkpoints", "alexnet_best.pth"), "alexnet_best.pth"]:
        if os.path.exists(alt_p):
            best_ckpt = alt_p
            break
start_epoch = 1
best_acc = 0.0
train_losses, test_losses = [], []
train_accs, test_accs = [], []

# Đọc lại checkpoint nếu đã có từ trước (Resume feature)
if os.path.exists(best_ckpt):
    ckpt = torch.load(best_ckpt, map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    start_epoch = ckpt["epoch"] + 1
    best_acc = ckpt.get("best_acc", 0.0)
    h = ckpt.get("history", {})
    train_losses = h.get("train_loss", [])
    test_losses = h.get("test_loss", [])
    train_accs = h.get("train_acc", [])
    test_accs = h.get("test_acc", [])
    print(f"Đã nạp checkpoint tốt nhất! Tiếp tục từ Epoch {start_epoch:02d} đến {args.epochs:02d} (Best Acc: {best_acc:.2f}%)")

for epoch in range(start_epoch, args.epochs + 1):
    # Train 1 epoch
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for inputs, targets, _ in trainloader:
        inputs, targets = inputs.to(device, non_blocking=True), targets.to(device, non_blocking=True)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        _, preds = outputs.max(1)
        correct += preds.eq(targets).sum().item()
        total += targets.size(0)

    train_loss = running_loss / total
    train_acc = 100.0 * correct / total

    # Test
    model.eval()
    test_loss_sum, test_correct, test_total = 0.0, 0, 0
    with torch.no_grad():
        for inputs, targets, _ in testloader:
            inputs, targets = inputs.to(device, non_blocking=True), targets.to(device, non_blocking=True)
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            test_loss_sum += loss.item() * inputs.size(0)
            _, preds = outputs.max(1)
            test_correct += preds.eq(targets).sum().item()
            test_total += targets.size(0)

    test_loss = test_loss_sum / test_total
    test_acc = 100.0 * test_correct / test_total
    scheduler.step()

    train_losses.append(train_loss)
    test_losses.append(test_loss)
    train_accs.append(train_acc)
    test_accs.append(test_acc)

    is_best = (test_acc >= best_acc)
    if is_best:
        best_acc = test_acc

    # In kết quả theo mẫu yêu cầu
    print(f"Epoch {epoch:02d} | Train Loss: {train_loss:.3f} | Train Acc: {train_acc:.2f}% | Test Loss: {test_loss:.3f} | Test Acc: {test_acc:.2f}%")

    # Chỉ lưu checkpoint của epoch tốt nhất (Best Test Acc) trong suốt 50 epoch
    if is_best or epoch == 1:
        if epoch == 1 and not is_best:
            best_acc = test_acc
        ckpt_data = {
            "epoch": epoch,
            "model_name": "alexnet",
            "total_params": total_params,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "best_acc": best_acc,
            "history": {
                "train_loss": train_losses, "test_loss": test_losses,
                "train_acc": train_accs, "test_acc": test_accs
            }
        }
        torch.save(ckpt_data, best_ckpt)


## Bước 6: Trực quan hóa & Xuất kết quả (Plots & Metrics)

In [ ]:
os.makedirs(args.plots_dir, exist_ok=True)
epochs_range = range(1, len(train_losses) + 1)

# 1. Đường cong Loss & Accuracy (Thống nhất thang đo trực quan)
plt.figure(figsize=(14, 5))

# Subplot 1: Loss
plt.subplot(1, 2, 1)
plt.plot(epochs_range, train_losses, 'o-', label='Train Loss', color='#2b5c8f', lw=2, markersize=4)
plt.plot(epochs_range, test_losses, 's-', label='Test Loss', color='#e76f51', lw=2, markersize=4)
plt.title(f"Loss Curves - Custom AlexNet (BatchNorm)", fontweight="bold")
plt.xlabel('Epoch')
plt.ylabel('Loss (Label Smoothing = 0.1)')
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()

# Subplot 2: Accuracy (Trục y cố định 0% đến 100%)
plt.subplot(1, 2, 2)
plt.plot(epochs_range, train_accs, 'o-', label='Train Acc', color='#2a9d8f', lw=2, markersize=4)
plt.plot(epochs_range, test_accs, 's-', label='Test Acc', color='#e76f51', lw=2, markersize=4)
plt.title(f"Accuracy Curves - Custom AlexNet (BatchNorm)", fontweight="bold")
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.ylim(0, 100)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()

plt.tight_layout()
curve_path = os.path.join(args.plots_dir, "alexnet_curves.png")
plt.savefig(curve_path, dpi=300)
plt.show()

# 2. Nạp lại trọng số của epoch tốt nhất (Best Acc) để đánh giá ma trận nhầm lẫn
if os.path.exists(best_ckpt):
    best_ckpt_data = torch.load(best_ckpt, map_location=device, weights_only=False)
    model.load_state_dict(best_ckpt_data["model_state_dict"])
    print(f">> Đã nạp trọng số tốt nhất: Epoch {best_ckpt_data['epoch']:02d} (Best Acc: {best_ckpt_data['best_acc']:.2f}%)")

model.eval()
all_preds, all_targets = [], []
with torch.no_grad():
    for inputs, targets, _ in testloader:
        outputs = model(inputs.to(device))
        _, preds = outputs.max(1)
        all_preds.extend(preds.cpu().numpy())
        all_targets.extend(targets.numpy())

cm = confusion_matrix(all_targets, all_preds)
breed_names = [name.replace('_', ' ').title() for name in testset.class_names]

plt.figure(figsize=(15, 13))
im = plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
plt.title(f"HÌNH 5: MA TRẬN NHẦM LẪN (CONFUSION MATRIX) - CUSTOM ALEXNET (BATCHNORM) TRÊN 37 GIỐNG PETS", fontweight="bold", fontsize=12, pad=15)
cbar = plt.colorbar(im, fraction=0.046, pad=0.04)
cbar.set_label('Số lượng ảnh', fontweight='bold', fontsize=11)

tick_marks = np.arange(len(breed_names))
plt.xticks(tick_marks, breed_names, rotation=90, fontsize=8)
plt.yticks(tick_marks, breed_names, fontsize=8)

plt.xlabel('Nhãn Dự Đoán (Predicted Breed)', fontweight='bold', fontsize=11, labelpad=10)
plt.ylabel('Nhãn Thực Tế (Ground Truth Breed)', fontweight='bold', fontsize=11, labelpad=10)
plt.tight_layout()
cm_path = os.path.join(args.plots_dir, "alexnet_confusion_matrix.png")
plt.savefig(cm_path, dpi=300, bbox_inches='tight')
plt.show()

# 3. Xuất file tóm tắt chỉ số định lượng
metrics_summary = {
    "model_name": "Custom AlexNet (BatchNorm)",
    "total_params": total_params,
    "batch_size": args.batch_size,
    "epochs_trained": len(train_losses),
    "final_train_loss": round(train_losses[-1], 3) if train_losses else None,
    "final_train_acc": round(train_accs[-1], 2) if train_accs else None,
    "final_test_loss": round(test_losses[-1], 3) if test_losses else None,
    "final_test_acc": round(test_accs[-1], 2) if test_accs else None,
    "best_test_acc": round(best_acc, 2)
}
metrics_file = os.path.join(args.plots_dir, "alexnet_metrics.json")
with open(metrics_file, "w") as f:
    json.dump(metrics_summary, f, indent=2)
print(f"Đã lưu kết quả đối sánh tại: {metrics_file}")
